## Import libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn import utils

import sklearn.tree as skt
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn import preprocessing

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

from kaggle_environments import make


from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import lightgbm as lgb

import matplotlib.pyplot as plt
plt.style.use('ggplot')

In [ ]:
!mkdir /kaggle_simulations
!mkdir /kaggle_simulations/agent
!mkdir /kaggle_simulations/agent/saved_model

In [ ]:
!pip install pip --upgrade -q
!pip install kaggle-environments --upgrade -q

## Create dataset

In [ ]:
# Old data
data_loc1 = '../input/santaepisodedatalebro/SantaEpisodeData_lebroschar.parquet'
data1 = pd.read_parquet(data_loc1)
print(data1.shape)

# Top 15 LB data
data_loc2 = '../input/santa2020top15lb-data/SantaEpisodeData_top15LB.parquet'
data2 = pd.read_parquet(data_loc2)
print(data2.shape)

# Concatenate data
data = pd.concat([data1, data2], axis=0, ignore_index=True)
data.reset_index(drop=True, inplace=True)
print(data.shape)

In [ ]:

# Top 15 LB data
data_loc = '../input/santa2020top15lb-data/SantaEpisodeData_top15LB.parquet'
data = pd.read_parquet(data_loc)
print(data.shape)

In [ ]:

# Old data
data_loc = '../input/santaepisodedatalebro/SantaEpisodeData_lebroschar.parquet'
data = pd.read_parquet(data_loc)
print(data.shape)

In [ ]:
data['game_progress'] = data['round_num'] / 2000
data['total_pulls'] = data['n_pulls_self'] + data['n_pulls_opp']
data['self_success_ratio'] = np.where(data['n_pulls_self']!=0, data['n_success_self'] / data['n_pulls_self'], 0)
data['exp_totSuccess_ratio'] = np.where(data['total_pulls']!=0, (data['n_success_self']*2) / data['total_pulls'], 0)

data.shape

In [ ]:
data.isnull().sum()

## Train | Test Split

In [ ]:
TRAIN_FEATS = ['round_num', 'n_pulls_self', 'n_success_self', 'n_pulls_opp', 'game_progress', 
               'total_pulls', 'self_success_ratio', 'exp_totSuccess_ratio']
TARGET_COL = 'payout'

data = data.fillna(0)

X = data[TRAIN_FEATS]

y = data[TARGET_COL].values


X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.1,random_state=101)

## Quick model

In [ ]:
data1 = data[data['round_num']<=500]
data2 = data[(data['round_num']>500) & (data['round_num']<=1000)]
data3 = data[(data['round_num']>1000) & (data['round_num']<=1500)]
data4 = data[data['round_num']>1500]

print(data1.shape)
print(data2.shape)
print(data3.shape)
print(data4.shape)

In [ ]:
%%time
model1 = skt.DecisionTreeRegressor(min_samples_leaf=100)

TRAIN_FEATS = [
    'round_num', 'n_pulls_self', 'n_success_self', 'n_pulls_opp', 
#                'game_progress', 'total_pulls', 'self_success_ratio', 'exp_totSuccess_ratio'
]

model1 = model1.fit(data1[TRAIN_FEATS].values, data1[TARGET_COL].values)

In [ ]:
%%time
model2 = skt.DecisionTreeRegressor(min_samples_leaf=100)

TRAIN_FEATS = [
    'round_num', 'n_pulls_self', 'n_success_self', 'n_pulls_opp', 
#                'game_progress', 'total_pulls', 'self_success_ratio', 'exp_totSuccess_ratio'
]

model2 = model2.fit(data2[TRAIN_FEATS].values, data2[TARGET_COL].values)

In [ ]:
%%time
model3 = skt.DecisionTreeRegressor(min_samples_leaf=100)

TRAIN_FEATS = [
    'round_num', 'n_pulls_self', 'n_success_self', 'n_pulls_opp', 
#                'game_progress', 'total_pulls', 'self_success_ratio', 'exp_totSuccess_ratio'
]

model3 = model3.fit(data3[TRAIN_FEATS].values, data3[TARGET_COL].values)

In [ ]:
%%time
model4 = skt.DecisionTreeRegressor(min_samples_leaf=100)

TRAIN_FEATS = [
    'round_num', 'n_pulls_self', 'n_success_self', 'n_pulls_opp', 
#                'game_progress', 'total_pulls', 'self_success_ratio', 'exp_totSuccess_ratio'
]

model4 = model4.fit(data4[TRAIN_FEATS].values, data4[TARGET_COL].values)

In [ ]:
# %%time
# #Setup a regressor
# hyper_params = {
#     'learning_rate': 0.05,
#     "max_depth": 8,
#     "n_estimators": 100,
#     "subsample": 0.8,
#     "random_state": 101,
# #     "verbose": 1
#     "verbosity": 2,
#     "n_jobs": -1
    
# }

# # reg = RandomForestRegressor()
# # reg = GradientBoostingRegressor(**hyper_params)
# reg = xgb.XGBRegressor(**hyper_params)

# # Fit model
# xgb_model = reg.fit(X_train,y_train)


In [ ]:
%%time
params = {'ccp_alpha': 470.50748016094076,
 'max_depth': 15,
 'max_features': 'auto',
 'min_samples_leaf': 0.04,
 'min_samples_split': 810,
 'random_state': 101,
 'splitter': 'best'}
model = skt.DecisionTreeRegressor(**params)

TRAIN_FEATS = [
    'round_num', 'n_pulls_self', 'n_success_self', 'n_pulls_opp', 
#                'game_progress', 'total_pulls', 'self_success_ratio', 'exp_totSuccess_ratio'
]

model = model.fit(data[TRAIN_FEATS].values, data[TARGET_COL].values)

In [ ]:
import pickle

filename = '/kaggle_simulations/agent/saved_model/model1.sav'
pickle.dump(model1, open(filename, 'wb'))

filename = '/kaggle_simulations/agent/saved_model/model2.sav'
pickle.dump(model2, open(filename, 'wb'))

filename = '/kaggle_simulations/agent/saved_model/model3.sav'
pickle.dump(model3, open(filename, 'wb'))

filename = '/kaggle_simulations/agent/saved_model/model4.sav'
pickle.dump(model4, open(filename, 'wb'))

In [ ]:
!ls /kaggle_simulations/agent/saved_model/

In [ ]:
rm /kaggle_simulations/agent/saved_model/model.sav

In [ ]:
%%writefile /kaggle_simulations/agent/main.py
"""Greedy agent that chooses machine based on maximum expected payout

Uses a trained decision tree model to consider the other player's movements
in the expected payout.

See my other kernel for methodology for generating training data:
https://www.kaggle.com/lebroschar/generate-training-data

"""
import pickle
import random

import numpy as np
import pandas as pd
import sklearn.tree as skt

# Parameters
FUDGE_FACTOR = 0.99
VERBOSE = False
# DATA_FILE = '/kaggle/input/training-data/training_data_201223.parquet'
TRAIN_FEATS = ['round_num', 'n_pulls_self', 'n_success_self', 'n_pulls_opp', 'game_progress', 
               'total_pulls', 'self_success_ratio', 'exp_totSuccess_ratio']
TARGET_COL = 'payout'

lastBandit = -1


def make_model(n):
    filename = '/kaggle_simulations/agent/saved_model/model{}.sav'.format(n)
    model = pickle.load(open(filename, 'rb'))
    return model


class GreedyStrategy:
    """Implements strategy to maximize expected value

    - Tracks estimated likelihood of payout ratio for each machine
    - Tracks number of pulls on each machine
    - Chooses machine based on maximum expected value
    
    
    """
    def __init__(self, n, n_machines):
        """Initialize and train decision tree model
        
        name, agent_num, 

        Args:
           name (str):   Name for the agent
           agent_num (int):   Assigned player number
           n_machines (int):   number of machines in the game
        
        """
        # Record inputs
#         self.name = name
#         self.agent_num = agent_num
        self.n_machines = n_machines
        
        # Initialize distributions for all machines
        self.n_pulls_self = np.array([0 for _ in range(n_machines)])
        self.n_success_self = np.array([0. for _ in range(n_machines)])
        self.n_pulls_opp = np.array([0 for _ in range(n_machines)])
        self.game_progress = np.array([0 for _ in range(n_machines)])
#         self.total_pulls = np.array([0 for _ in range(n_machines)])
#         self.self_success_ratio = np.array([0 for _ in range(n_machines)])
#         self.exp_totSuccess_ratio = np.array([0 for _ in range(n_machines)])

        # Track other players moves
        self.opp_moves = []
        
        # Track winnings
        self.last_reward_count = 0

        # Create model to predict expected reward
        self.model = make_model(n)
        
        
        # Predict expected reward
        features = np.zeros((self.n_machines, 4))
        features[:, 0] = len(self.opp_moves)
        features[:, 1] = self.n_pulls_self
        features[:, 2] = self.n_success_self
        features[:, 3] = self.n_pulls_opp
#         features[:, 4] = self.game_progress
#         features[:, 5] = self.total_pulls
#         features[:, 6] = self.self_success_ratio
#         features[:, 7] = self.exp_totSuccess_ratio
        self.predicts = self.model.predict(features)
        

    def __call__(self):
        """Choose machine based on maximum expected payout

        Returns:
           <result> (int):  index of machine to pull
        
        """
        # Otherwise, use best available
        est_return = self.predicts
        max_return = np.max(est_return)
        result = np.random.choice(np.where(
            est_return >= FUDGE_FACTOR * max_return)[0])
        
        if VERBOSE:
            print('  - Chose machine %i with expected return of %3.2f' % (
                int(result), est_return[result]))

        return int(result)
    
        
    def updateDist(self, curr_total_reward, last_m_indices, moi):
        """Updates estimated distribution of payouts"""
        # Compute last reward
        last_reward = curr_total_reward - self.last_reward_count
        self.last_reward_count = curr_total_reward
        if VERBOSE:
            print('Last reward: %i' % last_reward)

        if len(last_m_indices) == 2:
            # Update number of pulls for both machines
            m_index = last_m_indices[moi]
            opp_index = last_m_indices[(moi + 1) % 2]
            self.n_pulls_self[m_index] += 1
            self.n_pulls_opp[opp_index] += 1

            # Update number of successes
            self.n_success_self[m_index] += last_reward
            
            # Update opponent activity
            self.opp_moves.append(opp_index)
            

            # Update predictions for chosen machines
            self.predicts[[opp_index, m_index]] = self.model.predict(np.array([
                [
                    len(self.opp_moves),
                    self.n_pulls_self[opp_index],
                    self.n_success_self[opp_index],
                    self.n_pulls_opp[opp_index],
#                     len(self.opp_moves)/2000,
#                     self.n_pulls_self[opp_index]+self.n_pulls_opp[opp_index],
#                     (self.n_success_self[opp_index]/self.n_pulls_self[opp_index]) if self.n_pulls_self[opp_index]!=0 else 0,
#                     ((self.n_success_self[opp_index]*2)/(self.n_pulls_self[opp_index]+self.n_pulls_opp[opp_index])) if (self.n_pulls_self[opp_index]+self.n_pulls_opp[opp_index])!=0 else 0
                    
                ],
                [
                    len(self.opp_moves),
                    self.n_pulls_self[m_index],
                    self.n_success_self[m_index],
                    self.n_pulls_opp[m_index],
#                     len(self.opp_moves)/2000,
#                     self.n_pulls_self[m_index]+self.n_pulls_opp[m_index],
#                     (self.n_success_self[m_index]/self.n_pulls_self[m_index]) if self.n_pulls_self[m_index]!=0 else 0,
#                     ((self.n_success_self[m_index]*2)/(self.n_pulls_self[m_index]+self.n_pulls_opp[m_index])) if (self.n_pulls_self[m_index]+self.n_pulls_opp[m_index])!=0 else 0
                ]]).reshape(2,-1))
            

def agent(observation, configuration):
    global curr_agent, lastBandit
    
    if observation.step == 0:
        # Initialize agent
        curr_agent = GreedyStrategy(
#             'Mr. Agent %i' % observation['agentIndex'],
#             observation['agentIndex'],
            1,
            configuration['banditCount'])
        moi=0
    
    if observation.step == 501:
        # Initialize agent
        curr_agent = GreedyStrategy(
#             'Mr. Agent %i' % observation['agentIndex'],
#             observation['agentIndex'],
            2,
            configuration['banditCount'])
    
    if observation.step == 1001:
        # Initialize agent
        curr_agent = GreedyStrategy(
#             'Mr. Agent %i' % observation['agentIndex'],
#             observation['agentIndex'],
            3,
            configuration['banditCount'])
    
    if observation.step == 1501:
        # Initialize agent
        curr_agent = GreedyStrategy(
#             'Mr. Agent %i' % observation['agentIndex'],
#             observation['agentIndex'],
            4,
            configuration['banditCount'])
    
    if observation.step > 0:
        # Find self and opponent
        moi = int(lastBandit == observation.lastActions[1])
        if moi == 0: opp = 1
        else: opp = 0
    
    # Update payout ratio distribution with:
    curr_agent.updateDist(observation['reward'], observation['lastActions'], moi)
    
    lastBandit = curr_agent()

    return curr_agent()

In [ ]:
%%writefile randomAgent.py

import random

def random_agent(observation, configuration):
#     print(observation)
#     print(configuration)
    return random.randrange(configuration.banditCount)

In [ ]:
from kaggle_environments import make

env = make("mab", debug=True)

env.run(["/kaggle_simulations/agent/main.py", "randomAgent.py"])
env.render(mode="ipython", width=800, height=800)

In [ ]:
!cd /kaggle_simulations/agent && tar -czvf /kaggle/working/submit_custom_top15LB.tar.gz main.py saved_model

## Train model

In [ ]:
%%time

reg = skt.DecisionTreeRegressor(min_samples_leaf=40)

# Fit model
tree_model1 = reg.fit(X_train,y_train)

# Predict
pred = tree_model1.predict(X_test)

# Calculate Eval metrics
mse = mean_squared_error(y_test,pred)
mae = mean_absolute_error(y_test,pred)

print(reg.score(X_test,y_test))
print("MAE: {}".format(mae))
print("MSE: {}".format(mse))

In [ ]:
%%time

reg = skt.DecisionTreeRegressor(min_samples_leaf=200)

# Fit model
tree_model2 = reg.fit(X_train,y_train)

# Predict
pred = tree_model2.predict(X_test)

# Calculate Eval metrics
mse = mean_squared_error(y_test,pred)
mae = mean_absolute_error(y_test,pred)

print(reg.score(X_test,y_test))
print("MAE: {}".format(mae))
print("MSE: {}".format(mse))

In [ ]:
%%time

reg = skt.DecisionTreeRegressor(min_samples_leaf=500)

# Fit model
tree_model3 = reg.fit(X_train,y_train)

# Predict
pred = tree_model3.predict(X_test)

# Calculate Eval metrics
mse = mean_squared_error(y_test,pred)
mae = mean_absolute_error(y_test,pred)

print(reg.score(X_test,y_test))
print("MAE: {}".format(mae))
print("MSE: {}".format(mse))

In [ ]:
%%time

#Setup a regressor
hyper_params = {
    'learning_rate': 0.05,
    "max_depth": 8,
    "n_estimators": 100,
    "subsample": 0.8,
    "random_state": 101,
#     "verbose": 1
    "verbosity": 2,
    "n_jobs": -1
    
}

# reg = RandomForestRegressor()
# reg = GradientBoostingRegressor(**hyper_params)
reg = xgb.XGBRegressor(**hyper_params)

# Fit model
xgb_model = reg.fit(X_train,y_train)

# Predict
pred = xgb_model.predict(X_test)

# Calculate Eval metrics
mse = mean_squared_error(y_test,pred)
mae = mean_absolute_error(y_test,pred)

print(reg.score(X_test,y_test))
print("MAE: {}".format(mae))
print("MSE: {}".format(mse))

## Save model

In [ ]:
import pickle
filename = '/kaggle_simulations/agent/saved_model/tree_model1.sav'
pickle.dump(tree_model1, open(filename, 'wb'))

In [ ]:
import pickle
filename = '/kaggle_simulations/agent/saved_model/tree_model2.sav'
pickle.dump(tree_model2, open(filename, 'wb'))

In [ ]:
import pickle
filename = '/kaggle_simulations/agent/saved_model/tree_model3.sav'
pickle.dump(tree_model3, open(filename, 'wb'))

In [ ]:
import pickle
filename = '/kaggle_simulations/agent/saved_model/xgb_model.sav'
pickle.dump(xgb_model, open(filename, 'wb'))

In [ ]:
!ls /kaggle_simulations/agent/saved_model/

## Define competing agents

In [ ]:
%%writefile /kaggle_simulations/agent/tree_model1.py
"""Greedy agent that chooses machine based on maximum expected payout

Uses a trained decision tree model to consider the other player's movements
in the expected payout.

See my other kernel for methodology for generating training data:
https://www.kaggle.com/lebroschar/generate-training-data

"""
import pickle
import random

import numpy as np
import pandas as pd
import sklearn.tree as skt

# Parameters
FUDGE_FACTOR = 0.99
VERBOSE = False
# DATA_FILE = '/kaggle/input/training-data/training_data_201223.parquet'
TRAIN_FEATS = ['round_num', 'n_pulls_self', 'n_success_self', 'n_pulls_opp']
TARGET_COL = 'payout'


def make_model():
    filename = '/kaggle_simulations/agent/saved_model/tree_model1.sav'
    model = pickle.load(open(filename, 'rb'))
    return model


class GreedyStrategy:
    """Implements strategy to maximize expected value

    - Tracks estimated likelihood of payout ratio for each machine
    - Tracks number of pulls on each machine
    - Chooses machine based on maximum expected value
    
    
    """
    def __init__(self, name, agent_num, n_machines):
        """Initialize and train decision tree model

        Args:
           name (str):   Name for the agent
           agent_num (int):   Assigned player number
           n_machines (int):   number of machines in the game
        
        """
        # Record inputs
        self.name = name
        self.agent_num = agent_num
        self.n_machines = n_machines
        
        # Initialize distributions for all machines
        self.n_pulls_self = np.array([0 for _ in range(n_machines)])
        self.n_success_self = np.array([0. for _ in range(n_machines)])
        self.n_pulls_opp = np.array([0 for _ in range(n_machines)])

        # Track other players moves
        self.opp_moves = []
        
        # Track winnings
        self.last_reward_count = 0

        # Create model to predict expected reward
        self.model = make_model()
        
        # Predict expected reward
        features = np.zeros((self.n_machines, 4))
        features[:, 0] = len(self.opp_moves)
        features[:, 1] = self.n_pulls_self
        features[:, 2] = self.n_success_self
        features[:, 3] = self.n_pulls_opp
        self.predicts = self.model.predict(features)
        

    def __call__(self):
        """Choose machine based on maximum expected payout

        Returns:
           <result> (int):  index of machine to pull
        
        """
        # Otherwise, use best available
        est_return = self.predicts
        max_return = np.max(est_return)
        result = np.random.choice(np.where(
            est_return >= FUDGE_FACTOR * max_return)[0])
        
        if VERBOSE:
            print('  - Chose machine %i with expected return of %3.2f' % (
                int(result), est_return[result]))

        return int(result)
    
        
    def updateDist(self, curr_total_reward, last_m_indices):
        """Updates estimated distribution of payouts"""
        # Compute last reward
        last_reward = curr_total_reward - self.last_reward_count
        self.last_reward_count = curr_total_reward
        if VERBOSE:
            print('Last reward: %i' % last_reward)

        if len(last_m_indices) == 2:
            # Update number of pulls for both machines
            m_index = last_m_indices[self.agent_num]
            opp_index = last_m_indices[(self.agent_num + 1) % 2]
            self.n_pulls_self[m_index] += 1
            self.n_pulls_opp[opp_index] += 1

            # Update number of successes
            self.n_success_self[m_index] += last_reward
            
            # Update opponent activity
            self.opp_moves.append(opp_index)

            # Update predictions for chosen machines
            self.predicts[[opp_index, m_index]] = self.model.predict(np.array([
                [
                    len(self.opp_moves),
                    self.n_pulls_self[opp_index],
                    self.n_success_self[opp_index],
                    self.n_pulls_opp[opp_index]
                ],
                [
                    len(self.opp_moves),
                    self.n_pulls_self[m_index],
                    self.n_success_self[m_index],
                    self.n_pulls_opp[m_index]
                ]]).reshape(2,-1))
            

def agent(observation, configuration):
    global curr_agent
    
    if observation.step == 0:
        # Initialize agent
        curr_agent = GreedyStrategy(
            'Mr. Agent %i' % observation['agentIndex'],
            observation['agentIndex'],
            configuration['banditCount'])
    
    # Update payout ratio distribution with:
    curr_agent.updateDist(observation['reward'], observation['lastActions'])

    return curr_agent()

In [ ]:
%%writefile /kaggle_simulations/agent/tree_model3.py
"""Greedy agent that chooses machine based on maximum expected payout

Uses a trained decision tree model to consider the other player's movements
in the expected payout.

See my other kernel for methodology for generating training data:
https://www.kaggle.com/lebroschar/generate-training-data

"""
import pickle
import random

import numpy as np
import pandas as pd
import sklearn.tree as skt

# Parameters
FUDGE_FACTOR = 0.99
VERBOSE = False
# DATA_FILE = '/kaggle/input/training-data/training_data_201223.parquet'
TRAIN_FEATS = ['round_num', 'n_pulls_self', 'n_success_self', 'n_pulls_opp']
TARGET_COL = 'payout'


def make_model():
    filename = '/kaggle_simulations/agent/saved_model/tree_model3.sav'
    model = pickle.load(open(filename, 'rb'))
    return model


class GreedyStrategy:
    """Implements strategy to maximize expected value

    - Tracks estimated likelihood of payout ratio for each machine
    - Tracks number of pulls on each machine
    - Chooses machine based on maximum expected value
    
    
    """
    def __init__(self, name, agent_num, n_machines):
        """Initialize and train decision tree model

        Args:
           name (str):   Name for the agent
           agent_num (int):   Assigned player number
           n_machines (int):   number of machines in the game
        
        """
        # Record inputs
        self.name = name
        self.agent_num = agent_num
        self.n_machines = n_machines
        
        # Initialize distributions for all machines
        self.n_pulls_self = np.array([0 for _ in range(n_machines)])
        self.n_success_self = np.array([0. for _ in range(n_machines)])
        self.n_pulls_opp = np.array([0 for _ in range(n_machines)])

        # Track other players moves
        self.opp_moves = []
        
        # Track winnings
        self.last_reward_count = 0

        # Create model to predict expected reward
        self.model = make_model()
        
        # Predict expected reward
        features = np.zeros((self.n_machines, 4))
        features[:, 0] = len(self.opp_moves)
        features[:, 1] = self.n_pulls_self
        features[:, 2] = self.n_success_self
        features[:, 3] = self.n_pulls_opp
        self.predicts = self.model.predict(features)
        

    def __call__(self):
        """Choose machine based on maximum expected payout

        Returns:
           <result> (int):  index of machine to pull
        
        """
        # Otherwise, use best available
        est_return = self.predicts
        max_return = np.max(est_return)
        result = np.random.choice(np.where(
            est_return >= FUDGE_FACTOR * max_return)[0])
        
        if VERBOSE:
            print('  - Chose machine %i with expected return of %3.2f' % (
                int(result), est_return[result]))

        return int(result)
    
        
    def updateDist(self, curr_total_reward, last_m_indices):
        """Updates estimated distribution of payouts"""
        # Compute last reward
        last_reward = curr_total_reward - self.last_reward_count
        self.last_reward_count = curr_total_reward
        if VERBOSE:
            print('Last reward: %i' % last_reward)

        if len(last_m_indices) == 2:
            # Update number of pulls for both machines
            m_index = last_m_indices[self.agent_num]
            opp_index = last_m_indices[(self.agent_num + 1) % 2]
            self.n_pulls_self[m_index] += 1
            self.n_pulls_opp[opp_index] += 1

            # Update number of successes
            self.n_success_self[m_index] += last_reward
            
            # Update opponent activity
            self.opp_moves.append(opp_index)

            # Update predictions for chosen machines
            self.predicts[[opp_index, m_index]] = self.model.predict(np.array([
                [
                    len(self.opp_moves),
                    self.n_pulls_self[opp_index],
                    self.n_success_self[opp_index],
                    self.n_pulls_opp[opp_index]
                ],
                [
                    len(self.opp_moves),
                    self.n_pulls_self[m_index],
                    self.n_success_self[m_index],
                    self.n_pulls_opp[m_index]
                ]]).reshape(2,-1))
            

def agent(observation, configuration):
    global curr_agent
    
    if observation.step == 0:
        # Initialize agent
        curr_agent = GreedyStrategy(
            'Mr. Agent %i' % observation['agentIndex'],
            observation['agentIndex'],
            configuration['banditCount'])
    
    # Update payout ratio distribution with:
    curr_agent.updateDist(observation['reward'], observation['lastActions'])

    return curr_agent()

## Run simulations

In [ ]:
def print_rounds(file1, file2, N=3):
    env = make("mab", debug=True)
    p1_count=0
    p2_count=0
    print ('simulating...',N,'games')
    for i in range(N):
        game=env.run([file1, file2])
        p1_score = env.steps[-1][0]['reward']
        p2_score = env.steps[-1][1]['reward']
        if p1_score>p2_score:
            p1_count+=1
        elif p2_score>p1_score:
            p2_count+=1
        env.reset()
        z=i+1
        print(f"Round {i+1}: {p1_score} - {p2_score}")
#     print (p1_count,'for',z,round(p1_count/z,3),'.vs',round(p2_count/z,3))
    print("Champion wins: {} Challenger wins: {}".format(p1_count, p2_count))
    print("Champion win ratio: {} Challenger win ratio: {}".format(round(p1_count/z,3), round(p2_count/z,3)))
    print ('complete')
    points_est1=[]
    points_est2=[]
    
    for x in range(2000):
        #print (game[x][1]['reward'])
        z=x+1
        points_est1.append(game[x][0]['reward']/z)
        points_est2.append(game[x][1]['reward']/z)
        
    plt.plot(points_est1,label='champion')
    plt.plot(points_est2, label='challenger')
    plt.legend()
    plt.show()
    print (sum(points_est2)/len(points_est2))

In [ ]:
print('champion vs challenger')
print_rounds("/kaggle_simulations/agent/main.py", "randomAgent.py", 50)

## Save final model

In [ ]:
# Train with all data

In [ ]:
# Pickle to location
!mkdir /kaggle_simulations/agent/final_model

In [ ]:
# !cp /kaggle_simulations/agent/saved_model/model1.sav  /kaggle_simulations/agent/final_model/
# !mv /kaggle_simulations/agent/final_model/model1.sav /kaggle_simulations/agent/final_model/model.sav

In [ ]:
%%writefile /kaggle_simulations/agent/main.py
"""Greedy agent that chooses machine based on maximum expected payout

Uses a trained decision tree model to consider the other player's movements
in the expected payout.

See my other kernel for methodology for generating training data:
https://www.kaggle.com/lebroschar/generate-training-data

"""
import pickle
import random

import numpy as np
import pandas as pd
import sklearn.tree as skt

# Parameters
FUDGE_FACTOR = 0.99
VERBOSE = False
# DATA_FILE = '/kaggle/input/training-data/training_data_201223.parquet'
TRAIN_FEATS = ['round_num', 'n_pulls_self', 'n_success_self', 'n_pulls_opp']
TARGET_COL = 'payout'


def make_model():
    filename = '/kaggle_simulations/agent/final_model/model.sav'
    model = pickle.load(open(filename, 'rb'))
    return model


class GreedyStrategy:
    """Implements strategy to maximize expected value

    - Tracks estimated likelihood of payout ratio for each machine
    - Tracks number of pulls on each machine
    - Chooses machine based on maximum expected value
    
    
    """
    def __init__(self, name, agent_num, n_machines):
        """Initialize and train decision tree model

        Args:
           name (str):   Name for the agent
           agent_num (int):   Assigned player number
           n_machines (int):   number of machines in the game
        
        """
        # Record inputs
        self.name = name
        self.agent_num = agent_num
        self.n_machines = n_machines
        
        # Initialize distributions for all machines
        self.n_pulls_self = np.array([0 for _ in range(n_machines)])
        self.n_success_self = np.array([0. for _ in range(n_machines)])
        self.n_pulls_opp = np.array([0 for _ in range(n_machines)])
        self.game_progress = np.array([0 for _ in range(n_machines)])
        self.total_pulls = np.array([0 for _ in range(n_machines)])
        self.self_success_ratio = np.array([0 for _ in range(n_machines)])
        self.exp_totSuccess_ratio = np.array([0 for _ in range(n_machines)])

        # Track other players moves
        self.opp_moves = []
        
        # Track winnings
        self.last_reward_count = 0

        # Create model to predict expected reward
        self.model = make_model()
        
        # Predict expected reward
        features = np.zeros((self.n_machines, 4))
        features[:, 0] = len(self.opp_moves)
        features[:, 1] = self.n_pulls_self
        features[:, 2] = self.n_success_self
        features[:, 3] = self.n_pulls_opp
        features[:, 4] = self.game_progress
        features[:, 5] = self.total_pulls
        features[:, 6] = self.self_success_ratio
        features[:, 7] = self.exp_totSuccess_ratio
        self.predicts = self.model.predict(features)
        

    def __call__(self):
        """Choose machine based on maximum expected payout

        Returns:
           <result> (int):  index of machine to pull
        
        """
        # Otherwise, use best available
        est_return = self.predicts
        max_return = np.max(est_return)
        result = np.random.choice(np.where(
            est_return >= FUDGE_FACTOR * max_return)[0])
        
        if VERBOSE:
            print('  - Chose machine %i with expected return of %3.2f' % (
                int(result), est_return[result]))

        return int(result)
    
        
    def updateDist(self, curr_total_reward, last_m_indices):
        """Updates estimated distribution of payouts"""
        # Compute last reward
        last_reward = curr_total_reward - self.last_reward_count
        self.last_reward_count = curr_total_reward
        if VERBOSE:
            print('Last reward: %i' % last_reward)

        if len(last_m_indices) == 2:
            # Update number of pulls for both machines
            m_index = last_m_indices[self.agent_num]
            opp_index = last_m_indices[(self.agent_num + 1) % 2]
            self.n_pulls_self[m_index] += 1
            self.n_pulls_opp[opp_index] += 1

            # Update number of successes
            self.n_success_self[m_index] += last_reward
            
            # Update opponent activity
            self.opp_moves.append(opp_index)

            # Update predictions for chosen machines
            self.predicts[[opp_index, m_index]] = self.model.predict(np.array([
                [
                    len(self.opp_moves),
                    self.n_pulls_self[opp_index],
                    self.n_success_self[opp_index],
                    self.n_pulls_opp[opp_index],
                    len(self.opp_moves)/2000,
                    self.n_pulls_self[opp_index]+self.n_pulls_opp[opp_index],
                    (self.n_success_self[opp_index]/self.n_pulls_self[opp_index]) if self.n_pulls_self[opp_index]!=0 else 0,
                    ((self.n_success_self[opp_index]*2)/(self.n_pulls_self[opp_index]+self.n_pulls_opp[opp_index])) if (self.n_pulls_self[opp_index]+self.n_pulls_opp[opp_index])!=0 else 0
                ],
                [
                    len(self.opp_moves),
                    self.n_pulls_self[m_index],
                    self.n_success_self[m_index],
                    self.n_pulls_opp[m_index],
                    len(self.opp_moves)/2000,
                    self.n_pulls_self[m_index]+self.n_pulls_opp[m_index],
                    (self.n_success_self[m_index]/self.n_pulls_self[m_index]) if self.n_pulls_self[m_index]!=0 else 0,
                    ((self.n_success_self[m_index]*2)/(self.n_pulls_self[m_index]+self.n_pulls_opp[m_index])) if (self.n_pulls_self[m_index]+self.n_pulls_opp[m_index])!=0 else 0
                ]]).reshape(2,-1))
            

def agent(observation, configuration):
    global curr_agent
    
    if observation.step == 0:
        # Initialize agent
        curr_agent = GreedyStrategy(
            'Mr. Agent %i' % observation['agentIndex'],
            observation['agentIndex'],
            configuration['banditCount'])
    
    # Update payout ratio distribution with:
    curr_agent.updateDist(observation['reward'], observation['lastActions'])

    return curr_agent()

In [ ]:
!cd /kaggle_simulations/agent && tar -czvf /kaggle/working/submit1.tar.gz main.py saved_model